# MGMT298D: Science and Strategy of AI## Week 7: Word Embeddings & Transformers### UCLA Anderson School of Management

## PART A: Word Embeddings

### 1. Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.decomposition import PCAfrom sklearn.metrics.pairwise import cosine_similarityfrom sklearn.preprocessing import StandardScalerimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersfrom tensorflow.keras.datasets import reutersfrom tensorflow.keras.preprocessing.sequence import pad_sequencesimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')%matplotlib inline

### 2. Create Word Embeddings

In [ ]:
# Create synthetic embeddings grouped into semantic clustersnp.random.seed(42)# Define semantic clusters with their representative wordsclusters = {    'royalty': ['king', 'queen', 'prince', 'princess', 'emperor', 'empress', 'duke', 'duchess'],    'gender': ['man', 'woman', 'male', 'female', 'boy', 'girl', 'father', 'mother'],    'animals': ['dog', 'cat', 'bird', 'fish', 'lion', 'tiger', 'elephant', 'bear', 'wolf', 'horse'],    'colors': ['red', 'blue', 'green', 'yellow', 'orange', 'purple', 'black', 'white', 'pink', 'gray'],    'emotions': ['happy', 'sad', 'angry', 'joyful', 'furious', 'delighted', 'miserable', 'glad'],    'food': ['apple', 'bread', 'cheese', 'milk', 'chicken', 'rice', 'pasta', 'fish', 'meat', 'vegetable'],    'numbers': ['one', 'two', 'three', 'four', 'five', 'ten', 'hundred', 'thousand'],    'body': ['head', 'arm', 'leg', 'eye', 'hand', 'foot', 'heart', 'brain'],    'verbs': ['run', 'walk', 'jump', 'sit', 'stand', 'sleep', 'eat', 'drink'],    'nature': ['tree', 'flower', 'mountain', 'river', 'ocean', 'sky', 'cloud', 'sun', 'moon', 'star']}# Create embedding vectors where similar words are nearby in vector spaceembedding_dim = 50word_to_idx = {}embeddings = []idx_to_word = {}idx = 0for cluster_name, words in clusters.items():    # Create a base vector for this cluster    base = np.random.randn(embedding_dim) * 0.5        for word in words:        # Add small noise to base vector to create cluster around it        noise = np.random.randn(embedding_dim) * 0.1        embedding = base + noise        embedding = embedding / np.linalg.norm(embedding)  # Normalize                embeddings.append(embedding)        word_to_idx[word] = idx        idx_to_word[idx] = word        idx += 1embedding_matrix = np.array(embeddings)print(f'Created {len(word_to_idx)} word embeddings in {embedding_dim}-dimensional space')print(f'Clusters: {list(clusters.keys())}')print(f'Shape: {embedding_matrix.shape}')

### 3. Cosine Similarity

In [ ]:
# Define function to compute cosine similarity between wordsdef word_similarity(word1, word2):    if word1 not in word_to_idx or word2 not in word_to_idx:        return None    idx1, idx2 = word_to_idx[word1], word_to_idx[word2]    sim = cosine_similarity([embedding_matrix[idx1]], [embedding_matrix[idx2]])[0, 0]    return sim# Test similarity for semantic pairstest_pairs = [('king', 'queen'), ('king', 'car'), ('happy', 'sad'), ('dog', 'cat')]print('Word Similarities:')for word1, word2 in test_pairs:    sim = word_similarity(word1, word2)    print(f'{word1:10s} ~ {word2:10s}: {sim:.4f}')

### 4. Word Analogies

In [ ]:
# Implement analogy solving: find word where a - b + c ≈ ?def solve_analogy(a, b, c, top_k=5):    '''Solve: a is to b as c is to ?'''    if a not in word_to_idx or b not in word_to_idx or c not in word_to_idx:        return []        # Get vector for d where: d ≈ c - b + a    idx_a = word_to_idx[a]    idx_b = word_to_idx[b]    idx_c = word_to_idx[c]        vec_d = embedding_matrix[idx_c] - embedding_matrix[idx_b] + embedding_matrix[idx_a]    vec_d = vec_d / np.linalg.norm(vec_d)        # Find closest words (excluding a, b, c)    similarities = cosine_similarity([vec_d], embedding_matrix)[0]    sorted_indices = np.argsort(-similarities)        results = []    for idx in sorted_indices:        word = idx_to_word[idx]        if word not in [a, b, c]:            results.append((word, similarities[idx]))            if len(results) == top_k:                break        return results# Test analogiesprint('Word Analogies: a is to b as c is to ?\n')test_analogies = [    ('king', 'queen', 'man'),    ('man', 'woman', 'king'),    ('happy', 'sad', 'joyful')]for a, b, c in test_analogies:    results = solve_analogy(a, b, c)    print(f'{a} is to {b} as {c} is to:')    for word, sim in results[:3]:        print(f'  {word:15s} (similarity: {sim:.4f})')    print()

### 5. Embedding Visualization (PCA)

In [ ]:
# Select representative words from each cluster for visualizationselected_words = ['king', 'queen', 'man', 'woman', 'dog', 'cat', 'lion', 'tiger',                  'red', 'blue', 'green', 'happy', 'sad', 'angry', 'apple', 'bread',                  'tree', 'flower', 'sun', 'moon', 'run', 'walk', 'head', 'heart', 'eye']# Get embeddings for selected wordsselected_indices = [word_to_idx[w] for w in selected_words if w in word_to_idx]selected_embeddings = embedding_matrix[selected_indices]# Reduce to 2D with PCApca = PCA(n_components=2)embeddings_2d = pca.fit_transform(selected_embeddings)# Map words to their semantic groupsword_to_cluster = {}for cluster_name, words in clusters.items():    for word in words:        word_to_cluster[word] = cluster_name# Create color mappingunique_clusters = list(set(word_to_cluster.get(w, 'unknown') for w in selected_words))colors = sns.color_palette('husl', len(unique_clusters))color_map = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}# Plot embeddings with labelsfig, ax = plt.subplots(figsize=(12, 8))for i, word in enumerate([w for w in selected_words if w in word_to_idx]):    cluster = word_to_cluster.get(word, 'unknown')    ax.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1],               color=color_map[cluster], s=200, alpha=0.7)    ax.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]),               xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')ax.set_title('Word Embeddings Visualized with PCA (2D Projection)', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3)# Add legendfrom matplotlib.patches import Patchlegend_elements = [Patch(facecolor=color_map[cluster], label=cluster)                   for cluster in sorted(unique_clusters)]ax.legend(handles=legend_elements, loc='best', framealpha=0.9)plt.tight_layout()plt.show()print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')print(f'Total variance explained by 2 components: {sum(pca.explained_variance_ratio_):.2%}')

## PART B: Transformer Text Classification

### 6. Load Reuters Dataset

In [ ]:
# Load Reuters dataset and prepare sequencesprint('Loading Reuters dataset...')(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000)# Get word index for decoding (optional)word_index = reuters.get_word_index()reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])# Pad sequencesmax_len = 200x_train = pad_sequences(x_train, maxlen=max_len)x_test = pad_sequences(x_test, maxlen=max_len)print(f'Training set shape: {x_train.shape}')print(f'Test set shape: {x_test.shape}')print(f'Number of classes: {len(set(y_train))}')print(f'Train/test split: {len(y_train)} / {len(y_test)} samples')print(f'Sequence length: {max_len} tokens')print(f'Vocabulary size: 10000 words')

### 7. Build Transformer Block

In [ ]:
# Define TransformerBlock as Keras Layerclass TransformerBlock(layers.Layer):    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):        super(TransformerBlock, self).__init__()        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)        self.ffn = keras.Sequential([            layers.Dense(ff_dim, activation='relu'),            layers.Dense(embed_dim),        ])        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)        self.dropout1 = layers.Dropout(rate)        self.dropout2 = layers.Dropout(rate)    def call(self, inputs, training):        attn_output = self.att(inputs, inputs)        attn_output = self.dropout1(attn_output, training=training)        out1 = self.layernorm1(inputs + attn_output)                ffn_output = self.ffn(out1)        ffn_output = self.dropout2(ffn_output, training=training)        return self.layernorm2(out1 + ffn_output)# Define TokenAndPositionEmbedding layerclass TokenAndPositionEmbedding(layers.Layer):    def __init__(self, maxlen, vocab_size, embed_dim):        super(TokenAndPositionEmbedding, self).__init__()        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)    def call(self, x):        maxlen = tf.shape(x)[-1]        positions = tf.range(start=0, limit=maxlen, delta=1)        position_embeddings = self.pos_emb(positions)        token_embeddings = self.token_emb(x)        return token_embeddings + position_embeddingsprint('Transformer components defined:')print('- TransformerBlock: MultiHeadAttention + FeedForward with residual connections')print('- TokenAndPositionEmbedding: Token + positional embeddings')

### 8. Build Classifier

In [ ]:
# Build transformer classifier with configurable blocksdef build_transformer_classifier(num_transformer_blocks=1):    inputs = layers.Input(shape=(max_len,))    embedding_layer = TokenAndPositionEmbedding(max_len, 10000, 64)    x = embedding_layer(inputs)        for _ in range(num_transformer_blocks):        x = TransformerBlock(embed_dim=64, num_heads=2, ff_dim=64, rate=0.1)(x)        x = layers.GlobalAveragePooling1D()(x)    x = layers.Dropout(0.1)(x)    x = layers.Dense(20, activation='relu')(x)    outputs = layers.Dense(46, activation='softmax')(x)        model = keras.Model(inputs=inputs, outputs=outputs)    return model# Build 1-block modelmodel_1block = build_transformer_classifier(num_transformer_blocks=1)model_1block.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])print('1-Block Transformer Classifier:')model_1block.summary()

### 9. Train Transformer (1 Block)

In [ ]:
# Train 1-block transformerprint('Training 1-block Transformer...')history_1block = model_1block.fit(    x_train, y_train,    batch_size=32,    epochs=10,    validation_split=0.1,    verbose=0)# Evaluate on test settest_loss_1, test_acc_1 = model_1block.evaluate(x_test, y_test, verbose=0)print(f'\n1-Block Model Test Accuracy: {test_acc_1:.4f}')print(f'1-Block Model Test Loss: {test_loss_1:.4f}')print(f'Total Parameters: {model_1block.count_params():,}')

### 10. Training Curves

In [ ]:
# Plot training curvesfig, axes = plt.subplots(1, 2, figsize=(12, 4))# Accuracyaxes[0].plot(history_1block.history['accuracy'], label='Train', linewidth=2)axes[0].plot(history_1block.history['val_accuracy'], label='Validation', linewidth=2)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Accuracy')axes[0].set_title('1-Block Model: Accuracy')axes[0].legend()axes[0].grid(True, alpha=0.3)# Lossaxes[1].plot(history_1block.history['loss'], label='Train', linewidth=2)axes[1].plot(history_1block.history['val_loss'], label='Validation', linewidth=2)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Loss')axes[1].set_title('1-Block Model: Loss')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

### 11. Train Transformer (3 Blocks)

In [ ]:
# Build and train 3-block transformerprint('Training 3-block Transformer...')model_3blocks = build_transformer_classifier(num_transformer_blocks=3)model_3blocks.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])history_3blocks = model_3blocks.fit(    x_train, y_train,    batch_size=32,    epochs=10,    validation_split=0.1,    verbose=0)# Evaluate on test settest_loss_3, test_acc_3 = model_3blocks.evaluate(x_test, y_test, verbose=0)print(f'\n3-Block Model Test Accuracy: {test_acc_3:.4f}')print(f'3-Block Model Test Loss: {test_loss_3:.4f}')print(f'Total Parameters: {model_3blocks.count_params():,}')

### 12. Architecture Comparison

In [ ]:
# Compare architecturesfig, axes = plt.subplots(1, 2, figsize=(10, 4))# Test Accuracy Comparisonmodels_names = ['1-Block', '3-Block']accuracies = [test_acc_1, test_acc_3]colors = ['#1f77b4', '#ff7f0e']axes[0].bar(models_names, accuracies, color=colors, alpha=0.8, edgecolor='black')axes[0].set_ylabel('Test Accuracy')axes[0].set_title('Model Accuracy Comparison')axes[0].set_ylim([0, max(accuracies) * 1.15])axes[0].grid(True, alpha=0.3, axis='y')for i, (name, acc) in enumerate(zip(models_names, accuracies)):    axes[0].text(i, acc + 0.01, f'{acc:.4f}', ha='center', fontweight='bold')# Parameter Count Comparisonparam_counts = [model_1block.count_params(), model_3blocks.count_params()]axes[1].bar(models_names, param_counts, color=colors, alpha=0.8, edgecolor='black')axes[1].set_ylabel('Number of Parameters')axes[1].set_title('Model Complexity Comparison')axes[1].grid(True, alpha=0.3, axis='y')for i, (name, params) in enumerate(zip(models_names, param_counts)):    axes[1].text(i, params + 1000, f'{params:,}', ha='center', fontweight='bold', fontsize=9)plt.tight_layout()plt.show()print('\nArchitecture Summary:')print(f'1-Block:  {test_acc_1:.4f} accuracy, {model_1block.count_params():,} parameters')print(f'3-Block:  {test_acc_3:.4f} accuracy, {model_3blocks.count_params():,} parameters')print(f'Improvement: {(test_acc_3 - test_acc_1)*100:.2f} percentage points')